In [ ]:
# Setup: run this cell first (click it, then press Shift+Enter).
# It finds the lab folder so the imports below work, wherever you opened this notebook from.
import sys
from pathlib import Path
if "google.colab" in sys.modules and not any((p / "labkit").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    # In Colab, fetch a copy of the repo first so the toolkit and data are available.
    import os, subprocess
    subprocess.run(["git", "clone", "-q", "https://github.com/andrewcberkley/models-make-you-lose-control.git"], check=True)
    os.chdir("models-make-you-lose-control/notebooks")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "labkit").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import math
import numpy as np
import pandas as pd
from labkit.checks import check, check_value

print("Ready.")

Ready.


# Chapter 2. Error bars on a rate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewcberkley/models-make-you-lose-control/blob/main/solutions/ch02_error_bars_solution.ipynb)

## The situation

The safety case you'll critique after this chapter (in the Chapter 2 exercise), from **Supa Dupa Labs**, says *"Our monitor catches 95% of attacks (19 of 20 red-team attacks were flagged)."* But is that really 95%?

> **Why the name?** I named it as a nod to Missy Elliott's 1997 debut album *Supa Dupa Fly*, since a 95% catch rate sounds supa dupa until you put an error bar on it, and I named its coding agent, WorkIt, after "Work It" (2002), because whether the safety case holds up comes down to whether it's worth it.

## By the end you'll be able to

- explain why a rate measured on a small sample is uncertain, and roughly how uncertain it is
- show why the textbook error bar breaks down near 0% and 100%, which is where safety numbers usually sit
- compute a Wilson interval and check by simulation that it behaves
- restate a "19 of 20" claim honestly and say how many tests a firmer claim would need

## Why it matters

Assessments are full of rates (e.g., catch rates, refusal rates and failure rates), and they usually come from small samples close to 0% or 100%. Putting honest error bars on them is one of the quickest ways to show rigor in any write-up, and leaving them out is one of the most common weaknesses in safety cases.

## Part 1. The idea

If you run the same monitor against a fresh batch of 20 attacks, you'll get a slightly different count each time (just as flipping a coin 20 times doesn't always give 10 heads), so "19 of 20" is really one draw from a range of possible results.

The worked example below imagines two monitors, one that truly catches 80% of attacks and one that truly catches 95%, then runs 10,000 imaginary red-team campaigns of 20 attacks against each and counts how often each one scores 19 or 20.

In [ ]:
rng = np.random.default_rng(2)
for true_rate in [0.80, 0.95]:
    caught = rng.binomial(20, true_rate, size=10_000)   # attacks caught in each simulated campaign
    print(f"True catch rate {true_rate:.0%}: scored 19 or 20 out of 20 in {(caught >= 19).mean():.0%} of campaigns")

caught = rng.binomial(20, 0.95, size=10_000)
print("\nFor the 95% monitor, how many of 20 it caught across campaigns:")
print(pd.Series(caught).value_counts(normalize=True).sort_index(ascending=False).head(6).map("{:.0%}".format).to_string())

True catch rate 80%: scored 19 or 20 out of 20 in 7% of campaigns
True catch rate 95%: scored 19 or 20 out of 20 in 74% of campaigns

For the 95% monitor, how many of 20 it caught across campaigns:
20    36%
19    38%
18    19%
17     6%
16     1%
15     0%


**Read the output.** A monitor that catches only 80% of attacks still scores 19 or 20 about 7% of the time, while a genuine 95% monitor scores 18 about one time in five (and sometimes 17), so one batch of 20 can't separate those cases cleanly.

An **error bar**, or confidence interval, sums this up as the range of true rates that could plausibly have produced what you saw.

## Part 2. Your turn (the textbook error bar)

The interval most people learn first (called the Wald interval) is

$$\hat p \pm 1.96 \sqrt{\frac{\hat p (1-\hat p)}{n}}$$

where $\hat p = k/n$ is the rate you observed, with k successes out of n. The 1.96 comes from the normal distribution, since 95% of a normal curve lies within 1.96 standard errors of its center.

**Step 1.** Write `wald_interval(k, n, z)` so that it returns a pair `(low, high)`.

In [ ]:
def wald_interval(k, n, z=1.96):
    """Textbook interval: p_hat +/- z * sqrt(p_hat * (1 - p_hat) / n)."""
    p_hat = k / n
    half = z * math.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat - half, p_hat + half

In [ ]:
check("wald_interval", wald_interval, [((19, 20), (0.8545, 1.0455)), ((5, 10), (0.1901, 0.8099))],
      tol=1e-3, hint="half = z * math.sqrt(p_hat * (1 - p_hat) / n)")

Correct: wald_interval passed 2 checks.


<details><summary>Hint</summary>

```python
half = z * math.sqrt(p_hat * (1 - p_hat) / n)
return p_hat - half, p_hat + half
```
</details>

Now look at what this interval says in the two cases that matter most for safety work.

In [ ]:
from labkit import stats as kit
print("19 of 20 caught:", tuple(round(x, 3) for x in kit.wald_interval(19, 20)))
print(" 0 of 10 failed:", tuple(round(x, 3) for x in kit.wald_interval(0, 10)))

19 of 20 caught: (0.854, 1.046)
 0 of 10 failed: (0.0, 0.0)


For 19 of 20 the interval runs past 100%, which is impossible, and for 0 failures in 10 tests it says (0, 0), which claims certainty from just ten tests. Both happen because the formula assumes the observed rate is spread out like a symmetric bell curve, and near 0% or 100% it isn't.

## Part 3. Your turn (the Wilson interval)

The Wilson interval asks a better question (i.e., *which true rates would make our result unsurprising?*), and answering it gives an interval whose center is pulled a little toward 50% and whose width depends properly on n, so it always stays between 0% and 100%.

You don't need to derive it, because you can build it from three pieces.

- `z2 = z ** 2`
- **center** $= \dfrac{\hat p + z^2/(2n)}{1 + z^2/n}$
- **half-width** $= \dfrac{z}{1 + z^2/n} \sqrt{\dfrac{\hat p(1-\hat p)}{n} + \dfrac{z^2}{4n^2}}$

The interval runs from center minus half-width to center plus half-width, clipped to between 0 and 1.

**Step 2.** Fill in `center` and `half`.

In [ ]:
def wilson_interval(k, n, z=1.96):
    """Wilson score interval for k successes in n trials."""
    p_hat = k / n
    z2 = z ** 2
    center = (p_hat + z2 / (2 * n)) / (1 + z2 / n)
    half = (z / (1 + z2 / n)) * math.sqrt(p_hat * (1 - p_hat) / n + z2 / (4 * n ** 2))
    return max(0.0, center - half), min(1.0, center + half)

In [ ]:
check("wilson_interval", wilson_interval,
      [((19, 20), (0.7639, 0.9911)), ((0, 10), (0.0, 0.2775)), ((5, 10), (0.2366, 0.7634))],
      tol=1e-3, hint="Check the brackets: the whole numerator is divided by (1 + z2 / n).")

Correct: wilson_interval passed 3 checks.


<details><summary>Hint</summary>

```python
center = (p_hat + z2 / (2 * n)) / (1 + z2 / n)
half = (z / (1 + z2 / n)) * math.sqrt(p_hat * (1 - p_hat) / n + z2 / (4 * n ** 2))
```
</details>

For 19 of 20, Wilson gives about 76% to 99%. Notice how much that says, because the data fit a monitor that misses one attack in four just as well as one that misses one in a hundred.

## Part 4. Your turn (does the interval actually work?)

A 95% interval should contain the true rate in about 95% of studies, which is called **coverage**, and you can test it by simulation (i.e., pick a true rate, simulate many studies, compute the interval for each and count how often it contains the truth).

**Step 3.** Finish the loop so it counts the studies whose interval contains `true_rate`. In Python, `low <= x <= high` is True or False, and adding True to a number adds 1.

In [ ]:
def coverage(interval_fn, true_rate, n, sims=20000, seed=0):
    """Share of simulated studies whose interval contains the true rate."""
    rng = np.random.default_rng(seed)
    hits = 0
    for k in rng.binomial(n, true_rate, size=sims):
        low, high = interval_fn(int(k), n)
        hits += low <= true_rate <= high
    return hits / sims

In [ ]:
check("coverage", coverage,
      [((kit.wald_interval, 0.95, 20), 0.647), ((kit.wilson_interval, 0.95, 20), 0.926)],
      tol=0.01, hint="Replace pass with: hits += low <= true_rate <= high")

Correct: coverage passed 2 checks.


Here's the same comparison across a few situations, using the toolkit's version.

In [ ]:
print("true rate   tests   textbook   Wilson")
for true_rate, n in [(0.50, 20), (0.95, 20), (0.95, 100), (0.99, 100)]:
    print(f"{true_rate:>9.0%} {n:>7}   {kit.coverage(kit.wald_interval, true_rate, n):>8.1%} {kit.coverage(kit.wilson_interval, true_rate, n):>8.1%}")

true rate   tests   textbook   Wilson
      50%      20      95.9%    95.9%
      95%      20      64.7%    92.6%
      95%     100      87.7%    96.6%
      99%     100      64.1%    92.1%


**Read the output.** Near 50% both intervals are fine, but near 95% or 99% the textbook interval misses the truth up to a third of the time, while Wilson stays close to 95%. Wilson runs a little under at small samples, because no simple interval is exact there, and this is why it's the default for rates in safety work.

## Part 5. Check your understanding

Back to the safety case, which says *"Our monitor catches 95% of attacks (19 of 20 were flagged)."* If the monitor really does catch 95%, how many attacks would the lab need before it could show the catch rate is above 90%?

In [ ]:
low, high = kit.wilson_interval(19, 20)
print(f"19 of 20 -> plausible true catch rate {low:.0%} to {high:.0%}")

for n in range(20, 400, 20):
    k = round(0.95 * n)
    if kit.wilson_interval(k, n)[0] >= 0.90:
        print(f"With {k} of {n} caught (95%), the lower end first clears 90% at {kit.wilson_interval(k, n)[0]:.1%}")
        break

19 of 20 -> plausible true catch rate 76% to 99%
With 133 of 140 caught (95%), the lower end first clears 90% at 90.0%


### Your turn to explain it in your own words

Restate the 19-of-20 claim in two or three honest sentences that say what it really shows and what it would take to show more.

**Model answer.** Catching 19 of 20 red-team attacks is consistent with a true catch rate anywhere from about 76% to 99% (i.e., the 95% Wilson interval), so it doesn't establish 95%, and a monitor that catches only 80% of attacks would score 19 or 20 out of 20 about 7% of the time. To show the catch rate is above 90%, the lab would need roughly 140 attacks with 95% caught, and those attacks should be built against this monitor rather than reused from an older library.

## Recap

- A rate from n tests is one draw from a range, so always report an interval with it.
- The textbook interval fails near 0% and 100%, so use the Wilson interval instead.
- You can check any interval by simulating studies where you know the truth and measuring how often the interval contains it (i.e., its coverage).

## Self-check

<details><summary>1. Why does the textbook interval fail for 19 of 20?</summary>

It assumes the observed rate is spread symmetrically, so it can run past 100% and it gives zero width when there are no failures, and both of those break down near the edges.
</details>

<details><summary>2. What does Wilson do differently?</summary>

It asks which true rates would make the result unsurprising, which pulls its center toward 50%, makes its width depend properly on the sample size and keeps it between 0% and 100%.
</details>

<details><summary>3. What does "95% coverage" mean?</summary>

If you repeated the study many times, about 95% of the intervals you computed would contain the true rate.
</details>

**Next.** Do the Chapter 2 exercise, the safety-case critique, in `exercises/ch02_safety_case` (Claims 2 and 6 are where this chapter really pays off).

## Next

Chapter 3 turns these intervals into practice with your first Inspect evaluation, using fake tools and a scripted mock model.